# 01 · Attendance EDA (Data Understanding)

**Student Attendance Management System — Class M1**

This notebook is the *Data Understanding* phase of CRISP-DM. It explores the
cleaned attendance dataset to answer:

1. How much data do we have, and is it clean?
2. What is the overall attendance rate, and how is it split between present / late / absent?
3. How does attendance change week by week?
4. Which weekday has the most absences?
5. Which students are below the 75% threshold (feature 8)?

> The numbers here use the **same** functions as the live dashboard
> (`backend_django/api/stats.py`, re-exported by `analysis/metrics.py`), so the
> report and the app can never disagree.

**Before running:** from the project root run
`python backend_django/manage.py seed_demo`, then
`python -m analysis.export_data` and `python -m analysis.make_dataset`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Make the project root importable so `from analysis import ...` works inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from analysis import config, metrics  # noqa: E402

# Colours shared with the app (present / late / absent)
PRESENT, LATE, ABSENT, BRAND = "#0ca30c", "#fab219", "#d03b3b", "#2a78d6"
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False})
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(config.PROCESSED_DATA_PATH, parse_dates=["date"])
print(f"{len(df):,} attendance records · {df['student_code'].nunique()} students · "
      f"{df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## 1. Data quality

The cleaning step (`analysis/make_dataset.py`) already removed bad rows, so we only
confirm there are no missing values in the key columns and no impossible statuses.

In [ ]:
print("Missing values per column:")
print(df[["date", "student_code", "class_name", "subject_name", "status"]].isna().sum())
print("\nStatus values:", sorted(df["status"].unique()))
df["status"].value_counts()

## 2. Overall attendance rate

`rate = (present + late) / total` — late still counts as *attended*.

In [ ]:
summary = metrics.summarize(df)
overall = summary["overall"]
print(f"Overall attendance rate: {overall['rate']}%")
print(f"Present {overall['present']} · Late {overall['late']} · Absent {overall['absent']}")

counts = [overall["present"], overall["late"], overall["absent"]]
fig, ax = plt.subplots()
ax.bar(["Present", "Late", "Absent"], counts, color=[PRESENT, LATE, ABSENT])
for i, v in enumerate(counts):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
ax.set_title("Attendance records by status"); ax.set_ylabel("records")
plt.tight_layout(); plt.show()

## 3. Attendance rate over time (weekly)

This is the trend the **Trend Analysis** page (feature 9) shows.

In [ ]:
trend = metrics.trends(df, period="week")
series = pd.DataFrame(trend["series"])
series["period"] = pd.to_datetime(series["period"])

fig, ax = plt.subplots()
ax.plot(series["period"], series["rate"], marker="o", color=BRAND, linewidth=2)
ax.axhline(config.LOW_ATTENDANCE_THRESHOLD, color=ABSENT, linestyle="--", linewidth=1,
           label=f"{config.LOW_ATTENDANCE_THRESHOLD}% threshold")
ax.set_title("Attendance rate per week"); ax.set_ylabel("rate %"); ax.set_ylim(0, 100)
ax.legend(); fig.autofmt_xdate()
fig.savefig(config.FIGURES_DIR / "attendance_over_time.png", dpi=120, bbox_inches="tight")
plt.tight_layout(); plt.show()
for note in trend["insights"]:
    print("•", note)

## 4. Absence rate by weekday

Do students skip more on certain days?

In [ ]:
weekday = pd.DataFrame(trend["weekday"]).dropna(subset=["absent_rate"])
fig, ax = plt.subplots()
ax.bar(weekday["weekday"], weekday["absent_rate"], color=ABSENT)
ax.set_title("Absence rate by weekday"); ax.set_ylabel("absent %")
plt.tight_layout(); plt.show()
weekday[["weekday", "total", "absent", "absent_rate"]]

## 5. Low-attendance students (feature 8)

Students below the 75% threshold — the people a teacher should follow up with.

In [ ]:
low = pd.DataFrame(metrics.low_attendance(df, config.LOW_ATTENDANCE_THRESHOLD))
print(f"{len(low)} student(s) below {config.LOW_ATTENDANCE_THRESHOLD}%")
low[["student_code", "student_name", "total", "present", "late", "absent", "rate", "severity"]]

## Findings (fill in for the report)

- Overall attendance rate is **…%** across **…** records and **…** students.
- The weekly trend is **rising / falling / flat**; the biggest single-week change was **…**.
- **…day** has the highest absence rate.
- **…** students are below the 75% threshold and need follow-up.

➡️ Put these points and the saved figure (`reports/figures/attendance_over_time.png`)
into `reports/final_report.md` §2.